In [3]:
# install xgboost
!pip install xgboost


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


# Checking code works on subset of bins

In [5]:
# xgboost subset - to check code
import anndata as ad
import numpy as np

# load preprocessed data
rna_hvg = ad.read_h5ad("../preprocessing/outputs/rna_hvg.h5ad")
protein_data = ad.read_h5ad("../preprocessing/outputs/protein_data.h5ad")

# subset of bins (keeping all proteins/genes)
n_test = 3000   # adjust based on how fast you want the test run
rng = np.random.default_rng(42)
subset_idx = rng.choice(rna_hvg.n_obs, size=n_test, replace=False)

rna_hvg_test = rna_hvg[subset_idx].copy()
protein_data_test = protein_data[subset_idx].copy()

# save as test files
rna_hvg_test.write_h5ad("subset_test/rna_hvg_subset.h5ad")
protein_data_test.write_h5ad("subset_test/protein_data_subset.h5ad")

# new cross validation split for subset of bins
from cross_validation_split import cv_split

cv_split = cv_split(
    rna_path="subset_test/rna_hvg_subset.h5ad",
    output_path="subset_test",
    n_splits=5,
    random_state=42)

# run xgboost on subset of data
from xgboost_regression import xgboost_regression

xgboost_regression("subset_test/rna_hvg_subset.h5ad", "subset_test/protein_data_subset.h5ad",
                   "subset_test/cv_splits.json", "subset_test", device="cpu")


Fold 1/5, 
mean Pearson r=0.346, 
mean R2=0.122, 
mean RMSE=0.456
Fold 2/5, 
mean Pearson r=0.316, 
mean R2=0.070, 
mean RMSE=0.434
Fold 3/5, 
mean Pearson r=0.347, 
mean R2=0.118, 
mean RMSE=0.457
Fold 4/5, 
mean Pearson r=0.346, 
mean R2=0.118, 
mean RMSE=0.452
Fold 5/5, 
mean Pearson r=0.343, 
mean R2=0.107, 
mean RMSE=0.468

Overall mean Pearson r across all proteins: 0.340 ± 0.1253
Overall mean R² across all proteins: 0.107
Overall mean RMSE across all proteins: 0.453

Top 10 best-predicted proteins:
protein  mean_pearsonr  std_pearsonr  mean_r2  mean_rmse
   CD68       0.575359      0.029344 0.311193   0.394654
    MPO       0.529933      0.140856 0.211548   0.254753
  PD-L1       0.516884      0.034602 0.263875   0.181464
    CD4       0.515665      0.034484 0.256144   0.267665
   CD45       0.507543      0.029286 0.244130   0.347556
    TOX       0.496817      0.049538 0.201861   0.394949
 HLA-DR       0.490888      0.032133 0.225561   0.433295
   CD3e       0.478869      0.040

(   mean_pearsonr  mean_pearsonr_std  mean_r2  mean_rmse  \
 0       0.339739           0.125341   0.1069   0.453404   
 
                                           xgb_params  
 0  {'n_estimators': 300, 'max_depth': 4, 'learnin...  ,
        protein  mean_pearsonr  std_pearsonr   mean_r2  mean_rmse
 12        CD68       0.575359      0.029344  0.311193   0.394654
 26         MPO       0.529933      0.140856  0.211548   0.254753
 9        PD-L1       0.516884      0.034602  0.263875   0.181464
 35         CD4       0.515665      0.034484  0.256144   0.267665
 27        CD45       0.507543      0.029286  0.244130   0.347556
 32         TOX       0.496817      0.049538  0.201861   0.394949
 40      HLA-DR       0.490888      0.032133  0.225561   0.433295
 31        CD3e       0.478869      0.040829  0.217463   0.229538
 38        MGMT       0.461388      0.020873  0.200075   0.384452
 22        CD74       0.456484      0.023056  0.202588   0.789309
 41        CD14       0.445097      0.0

# Tuning

In [1]:
# tuning
from xgboost_tuning import xgboost_tuning

best_params, search_results_df = xgboost_tuning(
        rna_path="../preprocessing/outputs/rna_hvg.h5ad",
        pro_path="../preprocessing/outputs/protein_data.h5ad",
        cv_split_path="../preprocessing/outputs/cv_splits.json",
        n_proteins_to_sample=8,  # how many proteins to tune on,
        n_iter=20,  # number of random param combos to try per protein
        random_state=42)

KeyboardInterrupt: 

# Subset for multi-output comparison

In [10]:
import anndata as ad

rna_hvg = ad.read_h5ad("../preprocessing/outputs/rna_hvg.h5ad")
pro_data = ad.read_h5ad("../preprocessing/outputs/protein_data.h5ad")

In [11]:
import numpy as np

n_bins = 10000

rng = np.random.RandomState(42)
n_bins = min(n_bins, rna_hvg.n_obs)
idx = rng.choice(rna_hvg.n_obs, size=n_bins, replace=False)

rna_sub = rna_hvg[idx].copy()
pro_sub = pro_data[idx].copy()

In [12]:
rna_sub.write_h5ad("subset_test/rna_sub.h5ad")
pro_sub.write_h5ad("subset_test/pro_sub.h5ad")

# Multi-output regression comparison run

In [13]:
from xgboost_multioutput import run_loop_benchmark

loop_output = run_loop_benchmark(
    rna_hvg_path="subset_test/rna_sub.h5ad",
    protein_path="subset_test/pro_sub.h5ad",
    cv_split_path=None,
    xgb_params=None,
    fold_to_test=0)

Fold 0: running per-protein loop (44 fits)...


/Users/emilybeermann/Library/Python/3.9/lib/python/site-packages/xgboost/core.py:158: UserWarning: [11:52:24] WARNING: /Users/runner/work/xgboost/xgboost/src/context.cc:196: XGBoost is not compiled with CUDA support.
  warnings.warn(smsg, UserWarning)
/Users/emilybeermann/Library/Python/3.9/lib/python/site-packages/xgboost/core.py:158: UserWarning: [11:52:41] WARNING: /Users/runner/work/xgboost/xgboost/src/context.cc:196: XGBoost is not compiled with CUDA support.
  warnings.warn(smsg, UserWarning)
/Users/emilybeermann/Library/Python/3.9/lib/python/site-packages/xgboost/core.py:158: UserWarning: [11:52:59] WARNING: /Users/runner/work/xgboost/xgboost/src/context.cc:196: XGBoost is not compiled with CUDA support.
  warnings.warn(smsg, UserWarning)
/Users/emilybeermann/Library/Python/3.9/lib/python/site-packages/xgboost/core.py:158: UserWarning: [11:53:17] WARNING: /Users/runner/work/xgboost/xgboost/src/context.cc:196: XGBoost is not compiled with CUDA support.
  warnings.warn(smsg, UserW

  loop time: 1784.1s, mean Pearson r: 0.3769


In [14]:
from xgboost_multioutput import run_multioutput_benchmark

mo_output = run_multioutput_benchmark(
    rna_hvg_path="subset_test/rna_sub.h5ad",
    protein_path="subset_test/pro_sub.h5ad",
    cv_split_path=None,
    xgb_params=None,
    fold_to_test=0,
    device="cpu")

Fold 0: running multi_output_tree on cpu (1 fit)...
  multi_output_tree time: 1838.3s, mean Pearson r: 0.3828


In [18]:
from xgboost_multioutput import compare_results

compared = compare_results(loop_output, mo_output).sort_values(by=["pearsonr_loop"], ascending=False)

Speedup: 1.0x
       protein  pearsonr_loop  pearsonr_multioutput  pearsonr_diff   r2_loop  \
33        PD-1       0.342089              0.310943      -0.031146  0.081656   
29        FIBR       0.469159              0.444178      -0.024982  0.180215   
7        CXCR5       0.441528              0.418765      -0.022763  0.180606   
13        CD44       0.460555              0.444937      -0.015618  0.212006   
23        SIRP       0.329141              0.315050      -0.014092  0.099287   
25        IDH1       0.345432              0.332042      -0.013390  0.105291   
26         MPO       0.607435              0.599495      -0.007940  0.354214   
4       CXCL13       0.308699              0.302055      -0.006644  0.075211   
36        MAP2       0.408707              0.404909      -0.003798  0.165428   
14         SMA       0.346526              0.343901      -0.002625  0.090550   
28        CD21       0.270338              0.267834      -0.002504  0.053566   
42        ICOS       0.305

In [19]:
compared

,protein,pearsonr_loop,pearsonr_multioutput,pearsonr_diff,r2_loop,r2_multioutput
26,MPO,0.607435,0.599495,-0.007940,0.354214,0.358631
12,CD68,0.558748,0.558645,-0.000104,0.295832,0.305996
32,TOX,0.545274,0.549272,0.003998,0.279334,0.292648
27,CD45,0.511952,0.525980,0.014028,0.240447,0.266569
35,CD4,0.497995,0.506344,0.008350,0.224124,0.237440
40,HLA-DR,0.492743,0.506597,0.013854,0.222349,0.244210
9,PD-L1,0.489263,0.493989,0.004727,0.223168,0.243666
31,CD3e,0.482122,0.505105,0.022982,0.195956,0.251330
22,CD74,0.476965,0.487666,0.010701,0.218661,0.233985
38,MGMT,0.470519,0.472043,0.001523,0.204969,0.220493


# Cross validation run


In [ ]:
from xgboost_regression import xgboost_regression

results_df, per_protein_results_df = xgboost_regression(rna_hvg_path="../preprocessing/outputs/rna_hvg.h5ad",
                                                        protein_path="../preprocessing/outputs/protein_data.h5ad",
                                                        cv_split_path="../preprocessing/outputs/cv_splits.json",
                                                        out_path="results", xgb_params=None, device="cpu")

# Neighbourhood means on HVG - XGBoost test run on 3 protein subset

In [4]:
# subset
import anndata as ad

pro2 = ad.read_h5ad("preprocessing_outputs/protein_data_v2.h5ad")
pro3 = ad.read_h5ad("preprocessing_outputs/protein_data_v3.h5ad")

# subset proteins
pro2_sub = pro2[:, :3].copy()
pro3_sub = pro3[:, :3].copy()


# save subset
pro2_sub.write_h5ad("testsub/protein_data_test2.h5ad")
pro3_sub.write_h5ad("testsub/protein_data_test3.h5ad")


In [5]:
# no hop baseline
from xgboost_hvg import xgboost_hvg

results, per_protein, params = xgboost_hvg(
    rna_path="preprocessing_outputs/rna_hvg.h5ad",
    protein_path="testsub/protein_data_test2.h5ad",
    cv_split_path="preprocessing_outputs/cv_splits_patches.json",
    hop=None,
    out_path="testsub",
    device="cpu",
    xgb_params={"n_estimators": 50}
    )

Protein 1/3 took 13.6 seconds
Protein 2/3 took 13.9 seconds
Protein 3/3 took 13.9 seconds
Fold 1/5, 
mean Pearson r=0.190, 
mean R2=0.008, 
mean RMSE=0.956
Protein 1/3 took 11.2 seconds
Protein 2/3 took 15.2 seconds
Protein 3/3 took 15.2 seconds
Fold 2/5, 
mean Pearson r=0.139, 
mean R2=-0.065, 
mean RMSE=1.020
Protein 1/3 took 15.6 seconds
Protein 2/3 took 16.2 seconds
Protein 3/3 took 16.9 seconds
Fold 3/5, 
mean Pearson r=0.223, 
mean R2=-0.384, 
mean RMSE=1.097
Protein 1/3 took 15.4 seconds
Protein 2/3 took 15.1 seconds
Protein 3/3 took 15.7 seconds
Fold 4/5, 
mean Pearson r=0.299, 
mean R2=0.019, 
mean RMSE=0.971
Protein 1/3 took 19.6 seconds
Protein 2/3 took 19.4 seconds
Protein 3/3 took 19.9 seconds
Fold 5/5, 
mean Pearson r=0.220, 
mean R2=-0.022, 
mean RMSE=1.003

Overall mean Pearson r across all proteins: 0.214 ± 0.0276
Overall mean R² across all proteins: -0.089
Overall mean RMSE across all proteins: 1.009

Top 10 best-predicted proteins:
protein  mean_pearsonr  std_pearson

In [2]:
# add spatial features to rna data
from spatial_features.neighbourhood_means import neighbourhood_means

rna_hvg_aug_3 = neighbourhood_means(rna_path="preprocessing_outputs/rna_hvg.h5ad", out_path="results", hop=3)

In [2]:
from xgboost_hvg import xgboost_hvg

results, per_protein, params = xgboost_hvg(
    rna_path="spatial_features/results/hvg/rna_aug_hop3.h5ad",
    protein_path="testsub/protein_data_test.h5ad",
    cv_split_path="preprocessing_outputs/cv_splits_patches.json",
    out_path="testsub",
    device="cpu",
    xgb_params={"n_estimators": 50},
    hop=3)

Protein 1/3 took 40.0 seconds
Protein 2/3 took 57.2 seconds
Protein 3/3 took 80.9 seconds
Fold 1/5, 
mean Pearson r=0.209, 
mean R2=0.024, 
mean RMSE=0.948
Protein 1/3 took 74.4 seconds
Protein 2/3 took 91.5 seconds
Protein 3/3 took 94.0 seconds
Fold 2/5, 
mean Pearson r=0.196, 
mean R2=-0.050, 
mean RMSE=1.013
Protein 1/3 took 114.5 seconds
Protein 2/3 took 102.7 seconds
Protein 3/3 took 92.0 seconds
Fold 3/5, 
mean Pearson r=0.320, 
mean R2=-0.099, 
mean RMSE=0.992
Protein 1/3 took 85.4 seconds
Protein 2/3 took 84.9 seconds
Protein 3/3 took 83.3 seconds
Fold 4/5, 
mean Pearson r=0.413, 
mean R2=0.137, 
mean RMSE=0.912
Protein 1/3 took 112.9 seconds
Protein 2/3 took 115.9 seconds
Protein 3/3 took 104.2 seconds
Fold 5/5, 
mean Pearson r=0.338, 
mean R2=0.041, 
mean RMSE=0.972

Overall mean Pearson r across all proteins: 0.295 ± 0.0315
Overall mean R² across all proteins: 0.011
Overall mean RMSE across all proteins: 0.968

Top 10 best-predicted proteins:
protein  mean_pearsonr  std_pear

In [ ]:
# 30
rna_hvg_aug_30 = neighbourhood_means(rna_path="preprocessing_outputs/rna_hvg.h5ad", out_path="results", hop=30)

In [7]:
# 30
results, per_protein, params = xgboost_hvg(
    rna_path="spatial_features/results/hvg/rna_aug_hop30.h5ad",
    protein_path="testsub/protein_data_test.h5ad",
    cv_split_path="preprocessing_outputs/cv_splits_patches.json",
    out_path="testsub",
    device="cpu",
    xgb_params={"n_estimators": 50},
    hop=30)

Protein 1/3 took 28.2 seconds
Protein 2/3 took 36.5 seconds
Protein 3/3 took 35.9 seconds
Fold 1/5, 
mean Pearson r=0.216, 
mean R2=0.030, 
mean RMSE=0.947
Protein 1/3 took 981.5 seconds
Protein 2/3 took 40.5 seconds
Protein 3/3 took 32.6 seconds
Fold 2/5, 
mean Pearson r=0.405, 
mean R2=0.011, 
mean RMSE=0.979
Protein 1/3 took 44.2 seconds
Protein 2/3 took 44.6 seconds
Protein 3/3 took 45.5 seconds
Fold 3/5, 
mean Pearson r=0.355, 
mean R2=0.063, 
mean RMSE=0.924
Protein 1/3 took 43.3 seconds
Protein 2/3 took 43.5 seconds
Protein 3/3 took 43.2 seconds
Fold 4/5, 
mean Pearson r=0.334, 
mean R2=0.096, 
mean RMSE=0.930
Protein 1/3 took 54.4 seconds
Protein 2/3 took 56.6 seconds
Protein 3/3 took 53.5 seconds
Fold 5/5, 
mean Pearson r=0.463, 
mean R2=0.160, 
mean RMSE=0.911

Overall mean Pearson r across all proteins: 0.355 ± 0.0482
Overall mean R² across all proteins: 0.072
Overall mean RMSE across all proteins: 0.938

Top 10 best-predicted proteins:
protein  mean_pearsonr  std_pearsonr  

In [10]:
# 60
rna_hvg_aug_60 = neighbourhood_means(rna_path="preprocessing_outputs/rna_hvg.h5ad", out_path="results", hop=60)

In [12]:
# 60
results, per_protein, params = xgboost_hvg(
    rna_path="spatial_features/results/hvg/rna_aug_hop60.h5ad",
    protein_path="testsub/protein_data_test.h5ad",
    cv_split_path="preprocessing_outputs/cv_splits_patches.json",
    out_path="testsub",
    device="cpu",
    xgb_params={"n_estimators": 50},
    hop=60)

Protein 1/3 took 36.7 seconds
Protein 2/3 took 39.4 seconds
Protein 3/3 took 31.3 seconds
Fold 1/5, 
mean Pearson r=0.348, 
mean R2=0.105, 
mean RMSE=0.909
Protein 1/3 took 42.4 seconds
Protein 2/3 took 42.9 seconds
Protein 3/3 took 33.1 seconds
Fold 2/5, 
mean Pearson r=0.417, 
mean R2=0.042, 
mean RMSE=0.963
Protein 1/3 took 46.7 seconds
Protein 2/3 took 48.1 seconds
Protein 3/3 took 46.0 seconds
Fold 3/5, 
mean Pearson r=0.393, 
mean R2=0.105, 
mean RMSE=0.902
Protein 1/3 took 45.3 seconds
Protein 2/3 took 44.7 seconds
Protein 3/3 took 45.4 seconds
Fold 4/5, 
mean Pearson r=0.340, 
mean R2=0.112, 
mean RMSE=0.917
Protein 1/3 took 55.3 seconds
Protein 2/3 took 56.3 seconds
Protein 3/3 took 54.8 seconds
Fold 5/5, 
mean Pearson r=0.476, 
mean R2=0.193, 
mean RMSE=0.893

Overall mean Pearson r across all proteins: 0.395 ± 0.0691
Overall mean R² across all proteins: 0.112
Overall mean RMSE across all proteins: 0.917

Top 10 best-predicted proteins:
protein  mean_pearsonr  std_pearsonr  m

# XGBoost - no hop, with arcsinh

In [3]:
from xgboost_hvg import xgboost_hvg

results, per_protein, params = xgboost_hvg(
    rna_path="preprocessing_outputs/rna_hvg.h5ad",
    protein_path="testsub/protein_data_test3.h5ad",
    cv_split_path="preprocessing_outputs/cv_splits_patches.json",
    hop=None,
    out_path="testsub",
    device="cpu",
    xgb_params={"n_estimators": 50}
    )

Protein 1/3 took 13.6 seconds
Protein 2/3 took 13.7 seconds
Protein 3/3 took 14.2 seconds
Fold 1/5, 
mean Pearson r=0.130, 
mean R2=-0.037, 
mean RMSE=0.865
Protein 1/3 took 10.4 seconds
Protein 2/3 took 14.4 seconds
Protein 3/3 took 16.2 seconds
Fold 2/5, 
mean Pearson r=0.135, 
mean R2=-0.072, 
mean RMSE=1.212
Protein 1/3 took 15.7 seconds
Protein 2/3 took 15.4 seconds
Protein 3/3 took 17.0 seconds
Fold 3/5, 
mean Pearson r=0.192, 
mean R2=-0.770, 
mean RMSE=1.008
Protein 1/3 took 15.6 seconds
Protein 2/3 took 14.5 seconds
Protein 3/3 took 15.9 seconds
Fold 4/5, 
mean Pearson r=0.211, 
mean R2=-0.099, 
mean RMSE=0.899
Protein 1/3 took 13.2 seconds
Protein 2/3 took 19.1 seconds
Protein 3/3 took 18.1 seconds
Fold 5/5, 
mean Pearson r=0.131, 
mean R2=-0.028, 
mean RMSE=1.045

Overall mean Pearson r across all proteins: 0.160 ± 0.0619
Overall mean R² across all proteins: -0.201
Overall mean RMSE across all proteins: 1.006

Top 10 best-predicted proteins:
protein  mean_pearsonr  std_pears

# XGBoost with truncated SVD dimensionality reduction on 3 protein subset

In [ ]:
from spatial_features.neighbourhood_means import neighbourhood_means

neighbourhood_means()

In [5]:
# components 50
from xgboost_truncsvd import xgboost_svd

xgboost_svd(
    rna_path="preprocessing_outputs/rna_data.h5ad",
    protein_path="testsub/protein_data_test3.h5ad",
    cv_split_path="preprocessing_outputs/cv_splits_patches.json",
    hop=None,
    out_path="testsub",
    device="cpu",
    n_components=50,
    svd_random_state=0,
    xgb_params=None)

Protein 1/3 took 0.5 seconds
Protein 2/3 took 0.4 seconds
Protein 3/3 took 0.4 seconds
Fold 1/5, 
mean Pearson r=0.130, 
mean R2=-0.032, 
mean RMSE=0.863
Protein 1/3 took 0.5 seconds
Protein 2/3 took 0.4 seconds
Protein 3/3 took 1.2 seconds
Fold 2/5, 
mean Pearson r=0.251, 
mean R2=-0.041, 
mean RMSE=1.195
Protein 1/3 took 3.1 seconds
Protein 2/3 took 0.8 seconds
Protein 3/3 took 1.5 seconds
Fold 3/5, 
mean Pearson r=0.262, 
mean R2=-0.215, 
mean RMSE=0.865
Protein 1/3 took 2.9 seconds
Protein 2/3 took 1.7 seconds
Protein 3/3 took 1.0 seconds
Fold 4/5, 
mean Pearson r=0.362, 
mean R2=0.029, 
mean RMSE=0.845
Protein 1/3 took 2.8 seconds
Protein 2/3 took 0.7 seconds
Protein 3/3 took 1.7 seconds
Fold 5/5, 
mean Pearson r=0.374, 
mean R2=0.098, 
mean RMSE=0.980

Overall mean Pearson r across all proteins: 0.276 ± 0.0160
Overall mean R² across all proteins: -0.032
Overall mean RMSE across all proteins: 0.950
Mean SVD cumulative explained variance across folds: 0.032

Top 10 best-predicted p

(   mean_pearsonr  mean_pearsonr_std   mean_r2  mean_rmse  \
 0       0.275678           0.015971 -0.032316   0.949585   
 
    mean_svd_explained_var  n_svd_components   hop  
 0                0.032448                50  None  ,
   protein  mean_pearsonr  std_pearsonr   mean_r2  mean_rmse
 1   FOXP3       0.298131      0.136356  0.030260   0.960870
 0    synd       0.266571      0.120615 -0.147150   0.940918
 2    CD16       0.262331      0.063570  0.019941   0.946968,
 {'n_estimators': 550,
  'max_depth': 6,
  'learning_rate': 0.053,
  'subsample': 0.86,
  'colsample_bytree': 0.68,
  'tree_method': 'hist',
  'device': 'cpu',
  'random_state': 42,
  'n_jobs': -1})

In [3]:
# increase components to 300
from xgboost_truncsvd import xgboost_svd

xgboost_svd(
    rna_path="preprocessing_outputs/rna_data.h5ad",
    protein_path="testsub/protein_data_test2.h5ad",
    cv_split_path="preprocessing_outputs/cv_splits_patches.json",
    hop=None,
    out_path="testsub",
    device="cpu",
    n_components=50,
    svd_random_state=0,
    xgb_params=None)

Protein 1/3 took 2.3 seconds
Protein 2/3 took 0.4 seconds
Protein 3/3 took 0.5 seconds
Fold 1/5, 
mean Pearson r=0.191, 
mean R2=0.016, 
mean RMSE=0.952
Protein 1/3 took 0.6 seconds
Protein 2/3 took 0.4 seconds
Protein 3/3 took 1.1 seconds
Fold 2/5, 
mean Pearson r=0.231, 
mean R2=-0.035, 
mean RMSE=1.007
Protein 1/3 took 3.0 seconds
Protein 2/3 took 1.2 seconds
Protein 3/3 took 1.7 seconds
Fold 3/5, 
mean Pearson r=0.267, 
mean R2=-0.108, 
mean RMSE=0.996
Protein 1/3 took 1.5 seconds
Protein 2/3 took 1.9 seconds
Protein 3/3 took 1.2 seconds
Fold 4/5, 
mean Pearson r=0.380, 
mean R2=0.120, 
mean RMSE=0.920
Protein 1/3 took 2.8 seconds
Protein 2/3 took 0.7 seconds
Protein 3/3 took 1.6 seconds
Fold 5/5, 
mean Pearson r=0.335, 
mean R2=0.062, 
mean RMSE=0.962

Overall mean Pearson r across all proteins: 0.281 ± 0.0337
Overall mean R² across all proteins: 0.011
Overall mean RMSE across all proteins: 0.967
Mean SVD cumulative explained variance across folds: 0.032

Top 10 best-predicted pro

(   mean_pearsonr  mean_pearsonr_std   mean_r2  mean_rmse  \
 0       0.280796           0.033748  0.010955   0.967346   
 
    mean_svd_explained_var  n_svd_components   hop  
 0                0.032448                50  None  ,
   protein  mean_pearsonr  std_pearsonr   mean_r2  mean_rmse
 0    synd       0.325042      0.098174 -0.017356   0.948854
 1   FOXP3       0.274167      0.135445  0.022364   0.974660
 2    CD16       0.243178      0.037228  0.027855   0.978525,
 {'n_estimators': 550,
  'max_depth': 6,
  'learning_rate': 0.053,
  'subsample': 0.86,
  'colsample_bytree': 0.68,
  'tree_method': 'hist',
  'device': 'cpu',
  'random_state': 42,
  'n_jobs': -1})

In [3]:
# truncated SVD xgboost, 2000 HVG
from xgboost_truncsvd import xgboost_svd

xgboost_svd(
    rna_path="preprocessing_outputs/rna_hvg.h5ad",
    protein_path="testsub/protein_data_test3.h5ad",
    cv_split_path="preprocessing_outputs/cv_splits_patches.json",
    hop=None,
    out_path="testsub",
    device="cpu",
    n_components=50,
    svd_random_state=0,
    xgb_params=None)

Protein 1/3 took 0.5 seconds
Protein 2/3 took 0.5 seconds
Protein 3/3 took 0.3 seconds
Fold 1/5, 
mean Pearson r=0.116, 
mean R2=-0.047, 
mean RMSE=0.869
Protein 1/3 took 0.4 seconds
Protein 2/3 took 0.8 seconds
Protein 3/3 took 0.3 seconds
Fold 2/5, 
mean Pearson r=0.061, 
mean R2=-0.088, 
mean RMSE=1.221
Protein 1/3 took 1.5 seconds
Protein 2/3 took 0.8 seconds
Protein 3/3 took 1.2 seconds
Fold 3/5, 
mean Pearson r=0.157, 
mean R2=-0.757, 
mean RMSE=1.007
Protein 1/3 took 1.5 seconds
Protein 2/3 took 1.2 seconds
Protein 3/3 took 0.5 seconds
Fold 4/5, 
mean Pearson r=0.225, 
mean R2=-0.092, 
mean RMSE=0.895
Protein 1/3 took 0.5 seconds
Protein 2/3 took 1.5 seconds
Protein 3/3 took 0.6 seconds
Fold 5/5, 
mean Pearson r=0.167, 
mean R2=-0.026, 
mean RMSE=1.044

Overall mean Pearson r across all proteins: 0.145 ± 0.0585
Overall mean R² across all proteins: -0.202
Overall mean RMSE across all proteins: 1.007
Mean SVD cumulative explained variance across folds: 0.161

Top 10 best-predicted

(   mean_pearsonr  mean_pearsonr_std   mean_r2  mean_rmse  \
 0       0.144867           0.058486 -0.201849   1.007486   
 
    mean_svd_explained_var  n_svd_components   hop  
 0                0.160641                50  None  ,
   protein  mean_pearsonr  std_pearsonr   mean_r2  mean_rmse
 1   FOXP3       0.227578      0.078506 -0.034662   0.993508
 0    synd       0.103700      0.101441 -0.469257   1.026094
 2    CD16       0.103322      0.072400 -0.101629   1.002855,
 {'n_estimators': 550,
  'max_depth': 6,
  'learning_rate': 0.053,
  'subsample': 0.86,
  'colsample_bytree': 0.68,
  'tree_method': 'hist',
  'device': 'cpu',
  'random_state': 42,
  'n_jobs': -1})

In [4]:
# truncated SVD xgboost, 2000 HVG
from xgboost_truncsvd import xgboost_svd

xgboost_svd(
    rna_path="preprocessing_outputs/rna_hvg.h5ad",
    protein_path="testsub/protein_data_test2.h5ad",
    cv_split_path="preprocessing_outputs/cv_splits_patches.json",
    hop=None,
    out_path="testsub",
    device="cpu",
    n_components=50,
    svd_random_state=0,
    xgb_params=None)

Protein 1/3 took 0.6 seconds
Protein 2/3 took 0.5 seconds
Protein 3/3 took 0.4 seconds
Fold 1/5, 
mean Pearson r=0.164, 
mean R2=-0.006, 
mean RMSE=0.963
Protein 1/3 took 0.4 seconds
Protein 2/3 took 0.4 seconds
Protein 3/3 took 0.3 seconds
Fold 2/5, 
mean Pearson r=0.102, 
mean R2=-0.074, 
mean RMSE=1.025
Protein 1/3 took 1.5 seconds
Protein 2/3 took 0.8 seconds
Protein 3/3 took 1.3 seconds
Fold 3/5, 
mean Pearson r=0.197, 
mean R2=-0.370, 
mean RMSE=1.093
Protein 1/3 took 1.6 seconds
Protein 2/3 took 1.4 seconds
Protein 3/3 took 0.9 seconds
Fold 4/5, 
mean Pearson r=0.275, 
mean R2=0.026, 
mean RMSE=0.967
Protein 1/3 took 0.7 seconds
Protein 2/3 took 1.2 seconds
Protein 3/3 took 0.6 seconds
Fold 5/5, 
mean Pearson r=0.212, 
mean R2=-0.020, 
mean RMSE=1.002

Overall mean Pearson r across all proteins: 0.190 ± 0.0232
Overall mean R² across all proteins: -0.089
Overall mean RMSE across all proteins: 1.010
Mean SVD cumulative explained variance across folds: 0.161

Top 10 best-predicted 

(   mean_pearsonr  mean_pearsonr_std   mean_r2  mean_rmse  \
 0       0.190109             0.0232 -0.088771   1.009879   
 
    mean_svd_explained_var  n_svd_components   hop  
 0                0.160641                50  None  ,
   protein  mean_pearsonr  std_pearsonr   mean_r2  mean_rmse
 1   FOXP3       0.215526      0.069940 -0.031899   1.002730
 0    synd       0.195367      0.106887 -0.203352   1.019209
 2    CD16       0.159432      0.062440 -0.031061   1.007697,
 {'n_estimators': 550,
  'max_depth': 6,
  'learning_rate': 0.053,
  'subsample': 0.86,
  'colsample_bytree': 0.68,
  'tree_method': 'hist',
  'device': 'cpu',
  'random_state': 42,
  'n_jobs': -1})

# XGBoost with SVD, neighbourhood means of 3, 30 and 60 on subset of 3 proteins

In [2]:
from spatial_features.neighbourhood_means import neighbourhood_means

A = neighbourhood_means(rna_path="preprocessing_outputs/rna_data.h5ad", out_path="testsub", hop=3)

In [4]:
# with neighbourhood components
from xgboost_truncsvd import xgboost_svd

xgboost_svd(
    rna_path="preprocessing_outputs/rna_data.h5ad",
    protein_path="testsub/protein_data_test2.h5ad",
    cv_split_path="preprocessing_outputs/cv_splits_patches.json",
    hop=3,
    out_path="testsub",
    device="cpu",
    n_components=50,
    A=A)

Protein 1/3 took 2.9 seconds
Protein 2/3 took 0.7 seconds
Protein 3/3 took 1.3 seconds
Fold 1/5, 
mean Pearson r=0.264, 
mean R2=0.077, 
mean RMSE=0.919
Protein 1/3 took 0.9 seconds
Protein 2/3 took 3.2 seconds
Protein 3/3 took 0.8 seconds
Fold 2/5, 
mean Pearson r=0.286, 
mean R2=-0.011, 
mean RMSE=0.993
Protein 1/3 took 2.0 seconds
Protein 2/3 took 2.7 seconds
Protein 3/3 took 2.3 seconds
Fold 3/5, 
mean Pearson r=0.355, 
mean R2=0.109, 
mean RMSE=0.902
Protein 1/3 took 3.5 seconds
Protein 2/3 took 3.1 seconds
Protein 3/3 took 1.9 seconds
Fold 4/5, 
mean Pearson r=0.488, 
mean R2=0.232, 
mean RMSE=0.859
Protein 1/3 took 4.8 seconds
Protein 2/3 took 1.1 seconds
Protein 3/3 took 3.2 seconds
Fold 5/5, 
mean Pearson r=0.441, 
mean R2=0.192, 
mean RMSE=0.892

Overall mean Pearson r across all proteins: 0.367 ± 0.0611
Overall mean R² across all proteins: 0.120
Overall mean RMSE across all proteins: 0.913
Mean SVD cumulative explained variance across folds: 0.032

Top 10 best-predicted prot

(   mean_pearsonr  mean_pearsonr_std   mean_r2  mean_rmse  \
 0       0.366778             0.0611  0.119804   0.913292   
 
    mean_svd_explained_var  n_svd_components  hop  
 0                0.032448                50    3  ,
   protein  mean_pearsonr  std_pearsonr   mean_r2  mean_rmse
 0    synd       0.453161      0.149714  0.189059   0.852542
 2    CD16       0.325422      0.084714  0.083271   0.947506
 1   FOXP3       0.321752      0.205754  0.087082   0.939827,
 {'n_estimators': 550,
  'max_depth': 6,
  'learning_rate': 0.053,
  'subsample': 0.86,
  'colsample_bytree': 0.68,
  'tree_method': 'hist',
  'device': 'cpu',
  'random_state': 42,
  'n_jobs': -1})

In [1]:
# hop 30
from spatial_features.neighbourhood_means import neighbourhood_means

A30 = neighbourhood_means(rna_path="preprocessing_outputs/rna_data.h5ad", out_path="testsub", hop=30)

In [5]:
# 30
from xgboost_truncsvd import xgboost_svd

xgboost_svd(
    rna_path="preprocessing_outputs/rna_data.h5ad",
    protein_path="testsub/protein_data_test2.h5ad",
    cv_split_path="preprocessing_outputs/cv_splits_patches.json",
    hop=30,
    out_path="testsub",
    device="cpu",
    n_components=50,
    A=A30)

Protein 1/3 took 0.8 seconds
Protein 2/3 took 0.6 seconds
Protein 3/3 took 0.7 seconds
Fold 1/5, 
mean Pearson r=0.275, 
mean R2=0.076, 
mean RMSE=0.922
Protein 1/3 took 0.7 seconds
Protein 2/3 took 1.3 seconds
Protein 3/3 took 0.6 seconds
Fold 2/5, 
mean Pearson r=0.367, 
mean R2=0.016, 
mean RMSE=0.978
Protein 1/3 took 1.4 seconds
Protein 2/3 took 1.4 seconds
Protein 3/3 took 1.9 seconds
Fold 3/5, 
mean Pearson r=0.370, 
mean R2=0.128, 
mean RMSE=0.893
Protein 1/3 took 0.9 seconds
Protein 2/3 took 1.5 seconds
Protein 3/3 took 1.2 seconds
Fold 4/5, 
mean Pearson r=0.453, 
mean R2=0.183, 
mean RMSE=0.885
Protein 1/3 took 1.2 seconds
Protein 2/3 took 1.9 seconds
Protein 3/3 took 1.1 seconds
Fold 5/5, 
mean Pearson r=0.506, 
mean R2=0.212, 
mean RMSE=0.882

Overall mean Pearson r across all proteins: 0.394 ± 0.0508
Overall mean R² across all proteins: 0.123
Overall mean RMSE across all proteins: 0.912
Mean SVD cumulative explained variance across folds: 0.032

Top 10 best-predicted prote

(   mean_pearsonr  mean_pearsonr_std   mean_r2  mean_rmse  \
 0       0.394321           0.050826  0.123183   0.911834   
 
    mean_svd_explained_var  n_svd_components  hop  
 0                0.032448                50   30  ,
   protein  mean_pearsonr  std_pearsonr   mean_r2  mean_rmse
 0    synd       0.455070      0.081109  0.162440   0.866278
 1   FOXP3       0.397219      0.157025  0.127427   0.920242
 2    CD16       0.330673      0.074267  0.079683   0.948982,
 {'n_estimators': 550,
  'max_depth': 6,
  'learning_rate': 0.053,
  'subsample': 0.86,
  'colsample_bytree': 0.68,
  'tree_method': 'hist',
  'device': 'cpu',
  'random_state': 42,
  'n_jobs': -1})

In [2]:
# new parameters from tuning
from xgboost_truncsvd import xgboost_svd

xgboost_svd(
    rna_path="preprocessing_outputs/rna_data.h5ad",
    protein_path="testsub/protein_data_test2.h5ad",
    cv_split_path="preprocessing_outputs/cv_splits_patches.json",
    hop=30,
    out_path="testsub",
    device="cpu",
    n_components=50,
    A=A30)

Protein 1/3 took 2.5 seconds
Protein 2/3 took 2.0 seconds
Protein 3/3 took 2.8 seconds
Fold 1/5, 
mean Pearson r=0.257, 
mean R2=0.062, 
mean RMSE=0.929
Protein 1/3 took 2.9 seconds
Protein 2/3 took 3.2 seconds
Protein 3/3 took 1.2 seconds
Fold 2/5, 
mean Pearson r=0.292, 
mean R2=0.005, 
mean RMSE=0.983
Protein 1/3 took 6.5 seconds
Protein 2/3 took 7.3 seconds
Protein 3/3 took 7.4 seconds
Fold 3/5, 
mean Pearson r=0.347, 
mean R2=0.117, 
mean RMSE=0.898
Protein 1/3 took 3.0 seconds
Protein 2/3 took 6.6 seconds
Protein 3/3 took 4.5 seconds
Fold 4/5, 
mean Pearson r=0.433, 
mean R2=0.162, 
mean RMSE=0.896
Protein 1/3 took 4.6 seconds
Protein 2/3 took 5.6 seconds
Protein 3/3 took 5.5 seconds
Fold 5/5, 
mean Pearson r=0.473, 
mean R2=0.189, 
mean RMSE=0.895

Overall mean Pearson r across all proteins: 0.360 ± 0.0496
Overall mean R² across all proteins: 0.107
Overall mean RMSE across all proteins: 0.920
Mean SVD cumulative explained variance across folds: 0.032

Top 10 best-predicted prote

(   mean_pearsonr  mean_pearsonr_std   mean_r2  mean_rmse  \
 0       0.360457           0.049617  0.107015   0.920151   
 
    mean_svd_explained_var  n_svd_components  hop  
 0                0.032448                50   30  ,
   protein  mean_pearsonr  std_pearsonr   mean_r2  mean_rmse
 0    synd       0.426482      0.089473  0.145587   0.874930
 1   FOXP3       0.348018      0.175679  0.102276   0.933571
 2    CD16       0.306870      0.112992  0.073181   0.951953,
 {'n_estimators': 550,
  'max_depth': 6,
  'learning_rate': 0.019,
  'subsample': 0.76,
  'colsample_bytree': 0.8,
  'tree_method': 'hist',
  'device': 'cpu',
  'random_state': 42,
  'n_jobs': -1})